In [2]:
import pandas as pd
import numpy as np
import h5py
from datetime import datetime, timedelta
import sys

from matplotlib.colors import TwoSlopeNorm, LogNorm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

import geopandas as gpd

In [3]:
folder_path = "/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/hurricane_oct/"
sys.path.append(folder_path)
sys.path.append("/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/hurricane_oct/mobility_function/")
from mobility_function import analysis as ma
from importlib import reload
import mobility_function.analysis as ma
import mobility_function.hurricane_plotting as mhp
ma = reload(ma)
mhp = reload(mhp)

In [26]:
# Ms_i = ma.h5py_to_4d_array(folder_path+'/data/mobility/M_raw_20240909.h5')
Ms_i = ma.h5py_to_4d_array(folder_path+'/data/mobility/M_raw_20240923.h5')

In [23]:
GROUPS = {
    "Travel":       [1, 2,3, 4,5,12],      # e.g., local transit, taxis/rideshare, local roads/parking  intercity rail, air/water transport
    "Work & Professional": [9],         # e.g., offices, industrial/warehouse
    "Health":              [14],            # e.g., hospitals/clinics/pharmacies (if split, list them all)
    "Education":           [13],            # e.g., K-12, daycare, universities
    "Retail & Leisure":    [11,15],   # e.g., grocery, general retail, restaurants, arts/entertainment/fitness
    "Urban government":    [6,7,8,16],        # e.g., public admin, courts, safety (police/fire)
    "Utilities":           [0,10],        # e.g., utilities, waste mgmt/telecom
}

In [27]:
A, C, I, J = Ms_i.shape
names = list(GROUPS.keys())
G = len(names)
out = np.zeros((A, G, I, J), dtype=Ms_i.dtype) # number of days, number of groups, number of destination counties, number of origin counties

for g, name in enumerate(names):
    print(name)
    idx = GROUPS[name]
    block = Ms_i[:, idx, :, :]  # (A, k, I, J)
    out[:, g, :, :] = block.sum(axis=1)

Travel
Work & Professional
Health
Education
Retail & Leisure
Urban government
Utilities


In [28]:
with h5py.File("example_hh.h5", "w") as f: #example_h.h5#
    dset = f.create_dataset(
        "raw_visits", data=out,
        compression="gzip", compression_opts=4, 
        chunks=True                               
    )
    dset.attrs["dims"] = ["time", "group", "origin", "destination"]